# 7.3 隐私与安全

端侧部署大模型涉及用户数据隐私和模型知识产权保护。核心技术包括联邦学习、差分隐私、模型加密和安全推理。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 联邦学习（Federated Learning）

在端侧本地训练模型更新，仅上传梯度/参数差值到服务器聚合，原始数据不出端。

In [ ]:
class FederatedClient:
    """联邦学习客户端"""
    def __init__(self, model, data, lr=0.01, local_epochs=5):
        self.model = model
        self.data = data
        self.lr = lr
        self.local_epochs = local_epochs

    def train_local(self):
        """本地训练"""
        original_params = {n: p.clone() for n, p in self.model.named_parameters()}
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr)
        for _ in range(self.local_epochs):
            for x, y in self.data:
                loss = F.cross_entropy(self.model(x), y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        updates = {}
        for n, p in self.model.named_parameters():
            updates[n] = original_params[n] - p.data
        return updates

class FederatedServer:
    """联邦学习服务器"""
    def __init__(self, global_model):
        self.global_model = global_model

    def aggregate(self, client_updates, n_samples):
        """FedAvg聚合"""
        total_samples = sum(n_samples)
        with torch.no_grad():
            for name, param in self.global_model.named_parameters():
                weighted_update = torch.zeros_like(param)
                for updates, n in zip(client_updates, n_samples):
                    weighted_update += updates[name] * (n / total_samples)
                param.data -= weighted_update

class SimpleModel(nn.Module):
    def __init__(self, dim=32, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.net(x)

dim, n_classes = 32, 5
global_model = SimpleModel(dim, n_classes)
server = FederatedServer(global_model)

n_clients = 3
clients = []
for i in range(n_clients):
    local_model = SimpleModel(dim, n_classes)
    local_model.load_state_dict(global_model.state_dict())
    x = torch.randn(32 + i * 8, dim)
    y = torch.randint(0, n_classes, (32 + i * 8,))
    data = [(x, y)]
    clients.append(FederatedClient(local_model, data, lr=0.01, local_epochs=3))

n_rounds = 5
for round_idx in range(n_rounds):
    client_updates = []
    n_samples = []
    for client in clients:
        client.model.load_state_dict(global_model.state_dict())
        updates = client.train_local()
        client_updates.append(updates)
        n_samples.append(client.data[0][0].shape[0])
    server.aggregate(client_updates, n_samples)

x_test = torch.randn(64, dim)
y_test = torch.randint(0, n_classes, (64,))
with torch.no_grad():
    acc = (global_model(x_test).argmax(1) == y_test).float().mean()

print(f"=== 联邦学习 ===")
print(f"客户端数: {n_clients}")
print(f"通信轮数: {n_rounds}")
print(f"全局模型准确率: {acc:.4f}")
print(f"\n联邦学习核心:")
print(f"1. 原始数据不出端: 仅上传模型更新")
print(f"2. FedAvg聚合: 按样本量加权平均")
print(f"3. 适合端侧个性化: 每个客户端保留本地数据")

### 差分隐私（Differential Privacy）

In [ ]:
class DPSGD:
    """差分隐私SGD优化器"""
    def __init__(self, model, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0):
        self.model = model
        self.lr = lr
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm

    def step(self, loss):
        """差分隐私梯度更新"""
        loss.backward()
        total_norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), self.max_grad_norm)
        for p in self.model.parameters():
            if p.grad is not None:
                noise = torch.randn_like(p.grad) * self.noise_multiplier * self.max_grad_norm
                p.data -= self.lr * (p.grad + noise)
                p.grad = None

    def compute_epsilon(self, n_steps, delta=1e-5, batch_size=32, dataset_size=1000):
        """估算隐私预算ε"""
        q = batch_size / dataset_size
        sigma = self.noise_multiplier
        epsilon = q * np.sqrt(2 * n_steps * np.log(1/delta)) / sigma
        return epsilon

model_dp = SimpleModel(dim=32, n_classes=5)
dp_optimizer = DPSGD(model_dp, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0)

x = torch.randn(64, 32)
y = torch.randint(0, 5, (64,))

for _ in range(20):
    loss = F.cross_entropy(model_dp(x), y)
    dp_optimizer.step(loss)

with torch.no_grad():
    acc_dp = (model_dp(x).argmax(1) == y).float().mean()

model_no_dp = SimpleModel(dim=32, n_classes=5)
optimizer = torch.optim.SGD(model_no_dp.parameters(), lr=0.01)
for _ in range(20):
    loss = F.cross_entropy(model_no_dp(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    acc_no_dp = (model_no_dp(x).argmax(1) == y).float().mean()

epsilon = dp_optimizer.compute_epsilon(n_steps=20, batch_size=32, dataset_size=1000)

print(f"=== 差分隐私训练 ===")
print(f"噪声倍数: {dp_optimizer.noise_multiplier}")
print(f"梯度裁剪: {dp_optimizer.max_grad_norm}")
print(f"隐私预算ε: {epsilon:.4f}")
print(f"\n准确率对比:")
print(f"  无DP: {acc_no_dp:.4f}")
print(f"  有DP: {acc_dp:.4f}")
print(f"  精度损失: {(acc_no_dp - acc_dp)*100:.2f}%")
print(f"\n差分隐私权衡: ε越小隐私保护越强，但精度损失越大")
print(f"典型设置: ε=1~10, δ=1e-5")

### 模型水印

In [ ]:
class ModelWatermark:
    """模型水印: 在权重中嵌入不可见标识"""
    def __init__(self, key=42):
        self.key = key

    def embed(self, model, strength=0.001):
        """嵌入水印"""
        torch.manual_seed(self.key)
        watermark_bits = torch.randint(0, 2, (1024,))
        bit_idx = 0
        for name, param in model.named_parameters():
            if param.dim() >= 2:
                n_embed = min(param.numel() // 4, watermark_bits.shape[0] - bit_idx)
                if n_embed <= 0:
                    continue
                flat = param.data.flatten()
                for i in range(n_embed):
                    idx = i + bit_idx
                    if idx >= watermark_bits.shape[0]:
                        break
                    bit = watermark_bits[idx]
                    flat[i * 4] += strength * (1 if bit else -1)
                param.data = flat.reshape(param.shape)
                bit_idx += n_embed
        return watermark_bits

    def verify(self, model, watermark_bits, strength=0.001):
        """验证水印"""
        torch.manual_seed(self.key)
        bit_idx = 0
        correct = 0
        total = 0
        for name, param in model.named_parameters():
            if param.dim() >= 2:
                n_embed = min(param.numel() // 4, watermark_bits.shape[0] - bit_idx)
                if n_embed <= 0:
                    continue
                flat = param.data.flatten()
                for i in range(n_embed):
                    idx = i + bit_idx
                    if idx >= watermark_bits.shape[0]:
                        break
                    detected = flat[i * 4] > 0
                    expected = watermark_bits[idx].bool()
                    correct += (detected == expected).float().item()
                    total += 1
                bit_idx += n_embed
        return correct / max(total, 1)

model = SimpleModel(dim=32, n_classes=5)
watermarker = ModelWatermark(key=42)
bits = watermarker.embed(model, strength=0.01)
verify_rate = watermarker.verify(model, bits, strength=0.01)

x = torch.randn(32, 32)
y = torch.randint(0, 5, (32,))
with torch.no_grad():
    acc = (model(x).argmax(1) == y).float().mean()

print(f"=== 模型水印 ===")
print(f"水印验证率: {verify_rate:.2%}")
print(f"模型准确率: {acc:.4f} (水印对精度影响极小)")
print(f"\n模型水印用途:")
print(f"1. 模型溯源: 追踪模型来源")
print(f"2. 版权保护: 检测未授权使用")
print(f"3. 防止窃取: 验证模型是否被非法复制")